# Ejercicio 7 - Generalizacion entre lagos

Parte 2 del Laboratorio 4 (Machine Learning). Prueba si un modelo
entrenado con los datos de un lago sirve para predecir alta presencia de
cianobacteria en el otro. Dos experimentos cruzados con los tres modelos
ya elegidos: entrenar con Atitlan y evaluar con Amatitlan (Experimento A),
y entrenar con Amatitlan y evaluar con Atitlan (Experimento B).

Depende de los modelos ya entrenados (`notebooks/12_modelos.ipynb`) y del
diagnostico de identidad de lago que ya calculo la evaluacion
(`notebooks/13_evaluacion.ipynb`, `results/tables/diagnostico_identidad_lago.csv`).


## 0. Verificacion del trabajo previo (gate)


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.modelos import verificar_modelos, COLUMNAS_IDENTIDAD_LAGO
from src.evaluacion import verificar_evaluacion

resumen_modelos = verificar_modelos()
print(f"Gate de modelos correcto: {resumen_modelos['prueba']} observaciones de prueba.")

resumen_evaluacion = verificar_evaluacion()
print(f"Gate de evaluacion correcto: mejor modelo segun F2 es {resumen_evaluacion['mejor']}.")


Gate de modelos correcto: 147799 observaciones de prueba.
Gate de evaluacion correcto: mejor modelo segun F2 es gradient_boosting.


## 1. Un conjunto de predictores mas chico, por necesidad

Antes de correr los experimentos hay que quitar cuatro columnas:
`lago_amatitlan`, `lago_atitlan`, `x_utm` e `y_utm`. Son las mismas cuatro
que ya identifico `modelos.COLUMNAS_IDENTIDAD_LAGO` en el ejercicio
anterior. Los dos lagos ocupan rangos de coordenadas UTM que no se
solapan en absoluto, asi que dejar `x_utm`/`y_utm` no mide generalizacion
entre lagos: mide si el modelo memorizo en que rango de coordenadas vio
positivos, y el conjunto de prueba completo de un lago cae fuera de ese
rango por construccion.


In [2]:
from src.features import columnas_predictoras, leer_features
from src.validacion import COLUMNAS_EXCLUIDAS_GENERALIZACION, columnas_generalizacion

matriz = leer_features()
todas = columnas_predictoras(matriz)
reducidas = columnas_generalizacion(todas)

print(f"Predictores totales: {len(todas)}")
print(f"Excluidos para generalizacion: {sorted(COLUMNAS_EXCLUIDAS_GENERALIZACION)}")
print(f"Predictores que quedan: {len(reducidas)}")
print(sorted(reducidas))


Predictores totales: 17
Excluidos para generalizacion: ['lago_amatitlan', 'lago_atitlan', 'x_utm', 'y_utm']
Predictores que quedan: 13
['B03', 'B08', 'dia_anio_cos', 'dia_anio_sin', 'dist_centroide_m', 'dist_orilla_m', 'estacion_lluviosa', 'estacion_seca', 'frac_valida', 'mes', 'ndwi', 'ndwi_vecindad_3x3', 'ratio_B03_B08']


## 2. Los dos experimentos

El modelo clonado conserva los hiperparametros ya elegidos, incluido
`scale_pos_weight` del Gradient Boosting (calculado sobre la razon de
desbalance del dataset completo, no la del lago de entrenamiento): no se
retunea nada para maquillar el resultado, igual que el resto del
laboratorio. `class_weight="balanced"` (Regresion Logistica y Random
Forest) si se recalcula solo en cada `fit`, porque asi funciona ese
parametro en scikit-learn.


In [3]:
from src.validacion import escribir_metricas_generalizacion, validar_generalizacion

filas_generalizacion = validar_generalizacion()
escribir_metricas_generalizacion(filas_generalizacion)

tabla_generalizacion = pd.DataFrame(filas_generalizacion)
display(
    tabla_generalizacion[
        ["experimento", "modelo", "positivos_entrenamiento", "positivos_prueba",
         "recall", "precision", "f2", "roc_auc"]
    ]
)


,experimento,modelo,positivos_entrenamiento,positivos_prueba,recall,precision,f2,roc_auc
0,atitlan_a_amatitlan,regresion_logistica,7,6358,0.0,0.000000,0.000000,0.859366
1,amatitlan_a_atitlan,regresion_logistica,6358,7,1.0,0.000067,0.000337,0.916991
2,atitlan_a_amatitlan,random_forest,7,6358,0.0,0.000000,0.000000,0.600807
3,amatitlan_a_atitlan,random_forest,6358,7,1.0,0.000218,0.001087,0.971695
4,atitlan_a_amatitlan,gradient_boosting,7,6358,0.0,0.000000,0.000000,0.830666
5,amatitlan_a_atitlan,gradient_boosting,6358,7,1.0,0.000596,0.002973,0.985859


**Lectura:** los dos experimentos salen mal, y de formas opuestas.

El **Experimento A** (Atitlan -> Amatitlan) entrena con solo 7
observaciones positivas, las unicas que tiene Atitlan en todo el conjunto
de datos. Con tan poco ejemplo de que es "alta presencia", el modelo
aprende un umbral de decision demasiado exigente: al evaluarlo contra
Amatitlan, el recall es literalmente cero en los tres modelos, ni una
sola de las 6,358 celdas realmente positivas queda marcada como positiva.
El ROC-AUC no es igual de malo (0.60 a 0.86 segun el modelo): las
probabilidades si ordenan razonablemente bien las celdas de Amatitlan de
menor a mayor riesgo, pero el punto de corte aprendido en Atitlan queda
tan alto que ninguna celda de Amatitlan llega a cruzarlo.

El **Experimento B** (Amatitlan -> Atitlan) es el espejo. Entrenado con
los 6,358 positivos de Amatitlan, el modelo detecta las 7 celdas
positivas de Atitlan sin excepcion, recall de 1.0 en los tres modelos. El
costo es una inundacion de falsos positivos, con precision cayendo a un
rango de 0.007 a 0.06 por ciento: el modelo aprendio que "alta presencia"
se parece a los niveles de reflectancia tipicos de Amatitlan, y aplica
ese criterio sin ajustar a la escala completamente distinta de Atitlan,
donde practicamente todo queda por encima de ese umbral.


## 3-4. Comparacion contra el caso de ambos lagos mezclados

Para que la comparacion sea justa hay que usar el mismo conjunto reducido
de predictores en los dos casos. El diagnostico de identidad de lago del
ejercicio anterior ya entreno y evaluo los tres modelos sin esas mismas
cuatro columnas, pero sobre la particion aleatoria 70/30 que mezcla
observaciones de los dos lagos en entrenamiento y en prueba. Es la
comparacion correcta contra los experimentos de generalizacion: mismos
predictores, distinta fuente de las observaciones de entrenamiento.


In [4]:
diagnostico = pd.read_csv(ROOT / "results" / "tables" / "diagnostico_identidad_lago.csv")
f2_mezclado = diagnostico[diagnostico["metrica"] == "f2"].set_index("modelo")["sin_ubicacion"]

f2_generalizacion = (
    tabla_generalizacion[tabla_generalizacion["f2"] != "indefinido"]
    .assign(f2=lambda t: t["f2"].astype(float))
    .groupby("modelo")["f2"].agg(["min", "max"])
)

comparacion = pd.DataFrame({
    "f2_ambos_lagos_mezclados": f2_mezclado,
    "f2_generalizacion_min": f2_generalizacion["min"],
    "f2_generalizacion_max": f2_generalizacion["max"],
})
display(comparacion)


,f2_ambos_lagos_mezclados,f2_generalizacion_min,f2_generalizacion_max
modelo,,,
gradient_boosting,0.947143,0.0,0.002973
random_forest,0.926353,0.0,0.001087
regresion_logistica,0.593252,0.0,0.000337


**Lectura:** cuando el modelo ve observaciones de los dos lagos durante
el entrenamiento, aunque sea sin saber explicitamente a cual pertenece
cada una, el F2 se mantiene alto (0.59 a 0.94 segun el modelo). En cuanto
el entrenamiento se limita a un solo lago, el F2 se desploma a valores
practicamente nulos en los dos experimentos. La diferencia no esta en si
el modelo conoce la etiqueta del lago: esta en si alguna vez vio ejemplos
representativos del rango de condiciones del lago que despues tiene que
predecir.


## 5-6. Es esperable esta generalizacion?

No, con los datos de este laboratorio. Los dos experimentos muestran que
la capacidad de un modelo entrenado en un lago para predecir en el otro
es practicamente nula, aunque el ROC-AUC del Experimento B sugiera que
hay algo de estructura espectral compartida: las probabilidades si
ordenan razonablemente, el problema es la calibracion del umbral, no la
ausencia total de senal.

Amatitlan y Atitlan son sistemas muy distintos, con causas documentadas
por sus propias autoridades de manejo de cuenca (ver seccion 7 de la
Parte I):

- **Profundidad y volumen.** Atitlan es un lago volcanico, uno de los mas
  profundos de Centroamerica; Amatitlan es comparativamente playo. Un
  lago profundo con mucho volumen diluye los nutrientes que le entran;
  uno playo los concentra cerca de la superficie, donde las
  cianobacterias los aprovechan. Eso se traduce en una firma espectral de
  fondo muy distinta entre los dos lagos, incluso en observaciones sin
  ninguna floracion.
- **Presion urbana y aguas residuales.** La cuenca de Amatitlan esta
  dentro del area metropolitana de la Ciudad de Guatemala y recibe desde
  hace decadas una carga importante de aguas residuales e industriales;
  su autoridad de cuenca (AMSA) documenta un problema cronico de
  eutrofizacion. La cuenca de Atitlan, con autoridad AMSCLAE, tiene
  comparativamente menor densidad urbana y menor carga historica de
  aguas residuales sin tratar.
- **Escasez extrema de ejemplos positivos en Atitlan.** Con solo 7 celdas
  positivas en todo el periodo, cualquier modelo entrenado ahi aprende de
  una muestra demasiado pequena para capturar como se ve una floracion
  real, mas alla de esos 7 casos puntuales.

La consecuencia practica es clara: un modelo de este tipo no se puede
entrenar en un lago y desplegar en otro sin recalibrar, como minimo, el
punto de corte de decision a la escala de reflectancia propia del lago de
destino. Lo que si parece transferirse, segun el ROC-AUC del Experimento
B, es el orden relativo de riesgo dentro de cada lago, aunque no se
disponga de suficientes datos de Atitlan para confirmarlo con la misma
solidez que en Amatitlan.


## Verificacion final


In [5]:
from src.validacion import verificar_validacion

resumen_gate = verificar_validacion()
print(f"Verificacion correcta: {resumen_gate['bloques']['bloques_totales']} bloques en total.")
for ruta in resumen_gate["tablas"]:
    print(f"  {ruta}")


Verificacion correcta: 263 bloques en total.
  D:\Tareas\Data Science\Laboratorio 4 Parte 2\results\tables\bloques_espaciales.csv
  D:\Tareas\Data Science\Laboratorio 4 Parte 2\results\tables\metricas_validacion_espacial.csv
  D:\Tareas\Data Science\Laboratorio 4 Parte 2\results\tables\metricas_validacion_temporal.csv
  D:\Tareas\Data Science\Laboratorio 4 Parte 2\results\tables\metricas_generalizacion_lagos.csv
